# DC-Ops on Alif E8: fine-tune on the board camera (wave 3)

Continues training the deployed student (`v2_r160_kd`) on frames recorded from the E8's own camera
(slides of training-split components + venue background), with a replay of human-labelled crops.
Held-out slides are scored before and after. Needs a GPU runtime and the `HF_TOKEN` secret. ~10–15 min.

In [ ]:
from google.colab import userdata
import os, shutil
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download, HfApi
os.makedirs('/content/jobs', exist_ok=True)
for f in ['finetune_board.py', 'student.py']:
    shutil.copy(hf_hub_download('abhijitbetigeri/dcops-e8-jobs', f, force_download=True), f'/content/jobs/{f}')
data = hf_hub_download('abhijitbetigeri/dcops-e8-crops', 'board_ft.npz', repo_type='dataset', force_download=True)
init = hf_hub_download('abhijitbetigeri/dcops-e8-student', 'vision/v2_r160_kd/student.pt', force_download=True)
print(data, init)

In [ ]:
!python /content/jobs/finetune_board.py --data {data} --init {init} --out /content/v3_board_ft --epochs 25

In [ ]:
api = HfApi()
api.upload_folder(folder_path='/content/v3_board_ft', path_in_repo='vision/v3_board_ft', repo_id='abhijitbetigeri/dcops-e8-student')
print('pushed vision/v3_board_ft')